# Projekt · Woche 7 · Eure Wissensbasis
**AI Engineering: von LLMs zu Agenten** · linkit e.V. am KIT

Dieses Notebook baut die Wissensbasis eures Assistenten und misst, ob sie eure Testfragen abdeckt. Es nutzt dieselben Bausteine wie das Notebook der Woche 7, nur fertig programmiert.

**Ziel bis Woche 8:** Recall@5 für eure zehn Testfragen aus dem Steckbrief, plus eine kurze Notiz, was ihr bei Fehlschlägen geändert habt.

Ablauf: 1 Daten laden · 2 Chunken · 3 Einbetten und suchen · 4 Testfragen eintragen und messen · 5 Probe-Antwort mit Quellen · 6 Im Team-Repo speichern

In [ ]:
%pip install -q sentence-transformers==6.1.0 openai==3.24.0

## 1 · Daten laden

Ladet die ZIP-Datei eures Startpakets hoch (`Startpaket_A.zip` oder `Startpaket_B.zip`). Eigene Dokumente als `.md` oder `.txt` könnt ihr zusätzlich in den Ordner `daten/` legen (Dateisymbol links in Colab).

In [ ]:
import os, glob, zipfile, csv, re
import numpy as np

ORDNER = "daten"
if not os.path.isdir(ORDNER):
    from google.colab import files
    hochgeladen = files.upload()
    os.makedirs(ORDNER, exist_ok=True)
    for name in hochgeladen:
        if name.endswith(".zip"):
            zipfile.ZipFile(name).extractall(ORDNER)
        else:
            os.replace(name, os.path.join(ORDNER, name))

dateien = sorted(glob.glob(f"{ORDNER}/**/*.*", recursive=True))
print(len(dateien), "Dateien:")
for d in dateien:
    print("  ", d)

## 2 · Chunken

Markdown- und Textdateien werden entlang der Überschriften zerlegt, CSV-Dateien zeilenweise.

**Datenschutz-Regel:** Personenbezogene Listen gehören **nicht** in die Wissensbasis, die jede Frage durchsucht. Die Mitgliederliste aus Track A ist deshalb ausgeschlossen. Ab Woche 8 greift ein Tool mit klaren Regeln darauf zu.

In [ ]:
AUSSCHLIESSEN = {"mitglieder.csv"}     # bei Bedarf ergänzen

def chunk_struktur(markdown, dokument):
    chunks, pfad, zeilen, titel = [], {}, [], dokument
    def abschliessen():
        inhalt = "\n".join(zeilen).strip()
        if inhalt:
            chunks.append({"titel": titel, "text": inhalt})
    for zeile in markdown.splitlines():
        if zeile.startswith("#"):
            abschliessen()
            ebene = len(zeile) - len(zeile.lstrip("#"))
            pfad = {e: t for e, t in pfad.items() if e < ebene}
            pfad[ebene] = zeile.lstrip("#").strip()
            titel = dokument + " › " + " › ".join(pfad[e] for e in sorted(pfad))
            zeilen = []
        else:
            zeilen.append(zeile)
    abschliessen()
    return chunks

def chunk_csv(pfad, dokument):
    with open(pfad, encoding="utf-8") as f:
        zeilen = list(csv.DictReader(f))
    return [{"titel": f"{dokument} › Zeile {n}", "text": "; ".join(f"{k}: {v}" for k, v in z.items())}
            for n, z in enumerate(zeilen, start=1)]

CHUNKS = []
for d in dateien:
    name = os.path.basename(d)
    if name in AUSSCHLIESSEN or name.lower().startswith("readme"):
        continue
    if name.endswith((".md", ".txt")):
        CHUNKS += chunk_struktur(open(d, encoding="utf-8").read(), name)
    elif name.endswith(".csv"):
        CHUNKS += chunk_csv(d, name)

TEXTE = [c["titel"] + "\n" + c["text"] for c in CHUNKS]
print(len(CHUNKS), "Chunks aus", len({c['titel'].split(' › ')[0] for c in CHUNKS}), "Dokumenten")
print("Längster Chunk:", max(len(t) for t in TEXTE), "Zeichen (Modellgrenze etwa 2.000 Zeichen)")

## 3 · Einbetten und suchen

In [ ]:
from sentence_transformers import SentenceTransformer
modell = SentenceTransformer("intfloat/multilingual-e5-small")

E = modell.encode(["passage: " + t for t in TEXTE], normalize_embeddings=True)

def suche(frage, k=5):
    q = modell.encode(["query: " + frage], normalize_embeddings=True)[0]
    return list(np.argsort(E @ q)[::-1][:k])

for i in suche("Wie oft muss geschmiert werden?" if "wartungshandbuch.md" in str(dateien) else "Wie hoch ist der Mitgliedsbeitrag?", 3):
    print(CHUNKS[i]["titel"])

## 4 · Eure zehn Testfragen

Tragt die Fragen aus eurem Steckbrief ein. Der **Beleg** ist ein kurzer Textausschnitt, der genau so in einem Dokument steht und die Antwort enthält. So kann das Notebook automatisch prüfen, ob der richtige Chunk gefunden wurde.

Für Fragen, die nur ein Tool beantworten kann (etwa „Welche Tickets sind offen?“), oder für unlösbare Fragen tragt ihr `None` als Beleg ein. Sie werden beim Recall nicht mitgezählt.

In [ ]:
TESTFRAGEN_TEAM = [
    # (Frage, Beleg) – Beispiel 1 passt zu Track B, Beispiel 2 zu Track A. Das jeweils andere zeigt ⚠️.
    # Bitte durch eure eigenen zehn Fragen ersetzen.
    ("Wie oft muss das Spindellager der Drehmaschine nachgeschmiert werden?", "alle 500 Betriebsstunden"),
    ("Wie lange vorher muss zur Mitgliederversammlung eingeladen werden?", "mindestens zwei Wochen vorher"),
]

In [ ]:
def messen(fragen, k=5):
    zeilen, treffer_k, rr, n = [], 0, 0.0, 0
    for frage, beleg in fragen:
        if beleg is None:
            zeilen.append((frage, "—", "Tool-Frage oder unlösbar, nicht gezählt")); continue
        relevant = {i for i, t in enumerate(TEXTE) if beleg in t}
        if not relevant:
            zeilen.append((frage, "⚠️", f"Beleg „{beleg}“ steht in keinem Chunk. Tippfehler oder fehlt das Dokument?")); continue
        ranking = suche(frage, 20)
        rang = next((r for r, i in enumerate(ranking, start=1) if i in relevant), None)
        n += 1
        treffer_k += rang is not None and rang <= k
        rr += 1 / rang if rang else 0
        zeilen.append((frage, rang or ">20", CHUNKS[ranking[0]]["titel"]))
    for frage, rang, info in zeilen:
        print(f"{str(rang):>4}  {frage}\n      bester Treffer: {info}")
    if n:
        print(f"\nRecall@{k}: {treffer_k / n:.0%} · MRR: {rr / n:.2f} · gezählte Fragen: {n}")

messen(TESTFRAGEN_TEAM)

**Wenn eine Frage nicht gefunden wird**, prüft der Reihe nach:
1. Steht die Antwort überhaupt in einem Dokument? Sonst Dokument ergänzen oder Frage als unlösbar markieren.
2. Ist der Chunk zu lang oder zu kurz? Eine Überschrift mehr im Dokument kann helfen.
3. Fehlt dem Chunk Kontext (Maschinenname, Datum)? Dann den Titel oder das Dokument anpassen.
4. Nutzt die Frage andere Wörter als das Dokument? Dann hilft oft hybride Suche (Bonus B1 im Kursnotebook).

Notiert eure Änderungen hier:

*Änderungen und Ergebnis:*
- 

## 5 · Probe-Antwort mit Quellen (optional, braucht den API-Key)

In [ ]:
from openai import OpenAI
try:
    from google.colab import userdata
    API_KEY = userdata.get("KIT_TOOLBOX_KEY")
except ImportError:
    API_KEY = os.environ.get("KIT_TOOLBOX_KEY")
client = OpenAI(api_key=(API_KEY or "fehlt").strip(), base_url="https://ki-toolbox.scc.kit.edu/api/v1", max_retries=5)

SYSTEM = """Du bist der Assistent unseres Projekts. Beantworte Fragen ausschließlich mit den Quellen in <quellen>
und nenne die Quelle in eckigen Klammern, etwa [Q1]. Steht die Antwort nicht in den Quellen, sag das ehrlich."""

def beantworte(frage, k=4):
    besten = suche(frage, k)
    quellen = "\n".join(f'<quelle id="Q{n}" titel="{CHUNKS[i]["titel"]}">\n{CHUNKS[i]["text"]}\n</quelle>' for n, i in enumerate(besten, 1))
    antwort = client.chat.completions.create(model="kit.mistral-small-4-119b-a8b", max_tokens=400,
                                             messages=[{"role": "system", "content": SYSTEM},
                                                       {"role": "user", "content": f"<quellen>\n{quellen}\n</quellen>\n\nFrage: {frage}"}])
    return antwort.choices[0].message.content or ""

try:
    print(beantworte(TESTFRAGEN_TEAM[0][0]))
except Exception as e:
    print("Übersprungen:", type(e).__name__)

## 6 · Im Team-Repo speichern

*Datei → Kopie in GitHub speichern*, euer Team-Repo wählen, Commit-Nachricht zum Beispiel „Woche 7: Wissensbasis, Recall@5 = …“. Details stehen in `W07_Git-light.md`. Der API-Key wird dabei **nicht** gespeichert, er liegt nur in den Colab-Secrets.